# Text Representation

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

## Setup
Install the dependencies before running the examples and tasks. The Simpsons task dataset is included in `dataset/simpsons_script_lines.csv`. The earlier news lesson example uses its separate Kaggle dataset or a local `True.csv`.

In [ ]:
%pip install -q pandas scikit-learn nltk gensim kagglehub

In [2]:
from pathlib import Path
import nltk

nltk_data_folder = Path(".nlp_data/nltk")
nltk_data_folder.mkdir(parents=True, exist_ok=True)
nltk.data.path.insert(0, str(nltk_data_folder.resolve()))

for resource, location in {
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab/english",
}.items():
    try:
        nltk.data.find(location)
    except LookupError:
        if not nltk.download(resource, download_dir=str(nltk_data_folder), quiet=True):
            raise RuntimeError(f"Could not download NLTK resource: {resource}")

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [5]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [6]:
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [9]:
tfidf_vectorizer = TfidfVectorizer() # Create a TfidfVectorizer instance
vector_matrix = tfidf_vectorizer.fit_transform(data) # Fit and transform the documents into numerical vectors

In [10]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


In [11]:
import pandas as pd
import nltk
import numpy as np
import gensim

from nltk.tokenize import word_tokenize

In [ ]:
from pathlib import Path
import kagglehub

# Use a local True.csv for the course example, or download its Kaggle dataset.
news_csv = Path("True.csv")
if not news_csv.is_file():
    news_folder = Path(kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset"))
    matches = list(news_folder.rglob("True.csv"))
    if len(matches) != 1:
        raise FileNotFoundError("Place the news dataset's True.csv beside this notebook.")
    news_csv = matches[0]

df = pd.read_csv(news_csv)
df.head()

In [ ]:
tokens = [word_tokenize(text) for text in df["text"].dropna().astype(str)]
tokens = [sentence for sentence in tokens if sentence]

In [ ]:
w2v = gensim.models.Word2Vec(
    tokens, min_count=1, vector_size=100, window=5, sg=1,
    workers=1, seed=42, epochs=5,
)

In [ ]:
print("Cosine similarity between 'provide' and 'program' - Skip Gram : ",w2v.wv.similarity('provide', 'program'))

In [ ]:
print("words that similar to 'program' - Skip Gram : ",w2v.wv.most_similar('program'))

# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


#### Represent the Sentences and Compare Them
Use TF-IDF to represent the sentences as vectors, then calculate cosine similarity for every pair. The matrix diagonal compares each sentence with itself.

In [12]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]
sentence_labels = ["Sentence 1", "Sentence 2", "Sentence 3", "Sentence 4"]

sentence_vectorizer = TfidfVectorizer()
sentence_vectors = sentence_vectorizer.fit_transform(sentences)
sentence_similarity = cosine_similarity(sentence_vectors)

similarity_table = pd.DataFrame(
    sentence_similarity, index=sentence_labels, columns=sentence_labels
)
similarity_table.round(4)

,Sentence 1,Sentence 2,Sentence 3,Sentence 4
Sentence 1,1.0000,0.6469,0.3078,1.0000
Sentence 2,0.6469,1.0000,0.2252,0.6469
Sentence 3,0.3078,0.2252,1.0000,0.3078
Sentence 4,1.0000,0.6469,0.3078,1.0000


In [13]:
pair_scores = [
    (sentence_labels[i], sentence_labels[j], sentence_similarity[i, j])
    for i in range(len(sentences))
    for j in range(i + 1, len(sentences))
]
pair_table = pd.DataFrame(pair_scores, columns=["Sentence A", "Sentence B", "Similarity"])
pair_table = pair_table.sort_values("Similarity", ascending=False).reset_index(drop=True)
print("Most similar pair:", pair_table.loc[0, "Sentence A"], "and", pair_table.loc[0, "Sentence B"])
pair_table.round(4)

Most similar pair: Sentence 1 and Sentence 4


,Sentence A,Sentence B,Similarity
0,Sentence 1,Sentence 4,1.0000
1,Sentence 1,Sentence 2,0.6469
2,Sentence 2,Sentence 4,0.6469
3,Sentence 1,Sentence 3,0.3078
4,Sentence 3,Sentence 4,0.3078
5,Sentence 2,Sentence 3,0.2252


Sentences 1 and 4 contain the same words with the same counts, so their TF-IDF vectors match and their cosine similarity is approximately 1. This bag-of-words representation ignores word order and punctuation; matching vectors do not establish that sentence meanings are identical.

### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


#### Calculate the TF-IDF Weights
Each row is a sentence and each column is a vocabulary term. Higher weights indicate stronger importance within that sentence relative to this small corpus. Scikit-learn uses smoothed IDF and L2 normalization by default.

In [14]:
documents = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data",
]

task_tfidf = TfidfVectorizer()
tfidf_matrix = task_tfidf.fit_transform(documents)
feature_names = task_tfidf.get_feature_names_out()

tfidf_table = pd.DataFrame(
    tfidf_matrix.toarray(), columns=feature_names,
    index=["Sentence 1", "Sentence 2", "Sentence 3"],
)
tfidf_table.round(4)

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Sentence 1,0.0000,0.0000,0.0000,0.1895,0.3209,0.3209,0.2440,0.3209,0.4881,0.2440,0.4881,0.0000,0.2440,0.0000
Sentence 2,0.0000,0.4003,0.4003,0.2364,0.0000,0.0000,0.3044,0.0000,0.3044,0.3044,0.3044,0.0000,0.3044,0.4003
Sentence 3,0.5427,0.0000,0.0000,0.6411,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5427,0.0000,0.0000


#### Identify Important Words
Rank the nonzero terms in each sentence and display its top three. Terms appearing in fewer sentences receive higher IDF, while repeated words can also gain weight through their term frequency.

In [15]:
important_words = []
for sentence_label, weights in tfidf_table.iterrows():
    ranked_terms = sorted(
        [(term, weight) for term, weight in weights.items() if weight > 0],
        key=lambda item: (-item[1], item[0]),
    )
    for rank, (term, weight) in enumerate(ranked_terms[:3], start=1):
        important_words.append({
            "Sentence": sentence_label, "Rank": rank,
            "Word": term, "TF-IDF": weight,
        })

important_words_table = pd.DataFrame(important_words)
important_words_table.round({"TF-IDF": 4})

,Sentence,Rank,Word,TF-IDF
0,Sentence 1,1,of,0.4881
1,Sentence 1,2,science,0.4881
2,Sentence 1,3,fields,0.3209
3,Sentence 2,1,best,0.4003
4,Sentence 2,2,courses,0.4003
5,Sentence 2,3,this,0.4003
6,Sentence 3,1,data,0.6411
7,Sentence 3,2,analyze,0.5427
8,Sentence 3,3,scientists,0.5427


## Word2vec

### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)
```

#### Load the Simpsons Dataset
Use the CSV provided by the [course repository](https://github.com/MohammedMosuily/NLP_Lab5). Drop missing `spoken_words` values before preprocessing.

In [16]:
from pathlib import Path
import re
import gensim
from nltk.tokenize import word_tokenize

simpsons_csv = Path("dataset/simpsons_script_lines.csv")
if not simpsons_csv.is_file():
    raise FileNotFoundError("Run this notebook from lab5 with its dataset folder present.")

simpsons = pd.read_csv(simpsons_csv)
if "spoken_words" not in simpsons.columns:
    raise ValueError("The dataset must contain the spoken_words column.")

print(f"Dataset rows: {len(simpsons):,}")
print(f"Missing spoken_words: {simpsons['spoken_words'].isna().sum():,}")
simpsons = simpsons.dropna(subset=["spoken_words"]).copy()
simpsons[["spoken_words"]].head()

Dataset rows: 158,314
Missing spoken_words: 26,459


,spoken_words
0,"No, actually, it was a little of both. Sometim..."
1,Where's Mr. Bergstrom?
2,I don't know. Although I'd sure like to talk t...
3,That life is worth living.
4,The polls will be open from now until the end ...


#### Apply the Required Preprocessing
Use the lab's cleaning function: lowercase the dialogue, remove digits, and remove the listed punctuation characters. Then tokenize each dialogue line. Empty token lists are excluded from training.

In [17]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,”“.’$-]", "", text)
    return text

simpsons["clean_text"] = simpsons["spoken_words"].astype(str).apply(clean_text)
simpsons_tokens = [word_tokenize(text) for text in simpsons["clean_text"]]
simpsons_tokens = [tokens for tokens in simpsons_tokens if tokens]

print(f"Training dialogue lines: {len(simpsons_tokens):,}")
print(f"Training tokens: {sum(len(tokens) for tokens in simpsons_tokens):,}")
simpsons[["spoken_words", "clean_text"]].head()

Training dialogue lines: 131,839
Training tokens: 1,481,008


,spoken_words,clean_text
0,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...
1,Where's Mr. Bergstrom?,where's mr bergstrom?
2,I don't know. Although I'd sure like to talk t...,i don't know although i'd sure like to talk to...
3,That life is worth living.,that life is worth living
4,The polls will be open from now until the end ...,the polls will be open from now until the end ...


#### Train the Skip-Gram Word2Vec Model
Use the required `min_count=1`, `vector_size=100`, `window=5`, and `sg=1`. Train for five epochs with a fixed seed and one worker to reduce variation between runs. Similarity results depend on the corpus and trained embeddings.

In [18]:
Skip_gram_model = gensim.models.Word2Vec(
    sentences=simpsons_tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1,
    epochs=5,
    seed=42,
    workers=1,
)
print("Vocabulary size:", len(Skip_gram_model.wv))
print("Embedding dimensions:", Skip_gram_model.vector_size)

Vocabulary size: 44175
Embedding dimensions: 100


### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



#### Find Similar Words
Call `wv.most_similar()` for each name and show its 10 closest vocabulary words. The similarity column contains cosine similarities between word vectors, not probabilities.

In [19]:
similar_words = []
for name in ["homer", "marge", "bart"]:
    if name not in Skip_gram_model.wv:
        raise ValueError(f"Word is not in the model vocabulary: {name}")
    for rank, (word, similarity) in enumerate(
        Skip_gram_model.wv.most_similar(name, topn=10), start=1
    ):
        similar_words.append({
            "Query": name, "Rank": rank, "Similar word": word,
            "Cosine similarity": similarity,
        })

similar_words_table = pd.DataFrame(similar_words)
similar_words_table.round({"Cosine similarity": 4})

,Query,Rank,Similar word,Cosine similarity
0,homer,1,abe,0.8956
1,homer,2,marge,0.8535
2,homer,3,bart,0.8530
3,homer,4,eliza,0.8450
4,homer,5,bartholomew,0.8416
5,homer,6,grampa,0.8276
6,homer,7,allison,0.8215
7,homer,8,kirk,0.8202
8,homer,9,waylon,0.8154
9,homer,10,sweetheart,0.8151


### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

#### Find the Odd Word in Each List
Pass each group as a list to `wv.doesnt_match()`. The method selects the word least similar to the group's mean vector. This is an embedding-based result, so it may differ from a human's interpretation of the characters.

In [20]:
word_groups = [
    ["jimbo", "milhouse", "kearney"],
    ["nelson", "bart", "milhouse"],
    ["homer", "patty", "selma"],
]

odd_words = []
for group in word_groups:
    missing_words = [word for word in group if word not in Skip_gram_model.wv]
    if missing_words:
        raise ValueError(f"Words not found in the model vocabulary: {missing_words}")
    odd_word = Skip_gram_model.wv.doesnt_match(group)
    odd_words.append({"Word group": ", ".join(group), "Odd word": odd_word})

odd_words_table = pd.DataFrame(odd_words)
odd_words_table

,Word group,Odd word
0,"jimbo, milhouse, kearney",milhouse
1,"nelson, bart, milhouse",nelson
2,"homer, patty, selma",homer
